In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/cell_jepa_checkpoints", exist_ok=True)

In [ ]:
# Download the Severin PBMC dataset to Colab VM
!wget -O /content/severin_pbmc.zip "https://www.research-collection.ethz.ch/bitstreams/8689d69b-d916-4c8e-9b3f-2981c512b70b/download"

!mkdir -p /content/severin_data
!unzip -q /content/severin_pbmc.zip -d /content/severin_data

# See what's inside
!ls /content/severin_data/
!find /content/severin_data -name "*.tif" | head -20
!find /content/severin_data -type d | head -20

test background %

In [ ]:
import tifffile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

base = "/content/severin_data/DeepPhenotype_PBMC_ImageSet_YSeverin/Training"
import glob
files = glob.glob(f"{base}/**/*.tiff", recursive=True)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for i in range(5):
    img = tifffile.imread(files[i*200]).astype(np.float32)
    img_norm = img / img.max()

    # Row 1: Original 50x50 (all channels merged)
    composite = img_norm.mean(axis=2)
    axes[0, i].imshow(composite, cmap='gray')
    axes[0, i].set_title(f"50x50 original")
    axes[0, i].axis('off')

    # Row 2: Resized 224x224
    ch_resized = Image.fromarray(composite).resize((224, 224), Image.BILINEAR)
    axes[1, i].imshow(np.array(ch_resized), cmap='gray')
    axes[1, i].set_title(f"224x224 resized")
    axes[1, i].axis('off')

    # Row 3: Show the 14x14 patch grid on resized image
    ch_arr = np.array(ch_resized)
    axes[2, i].imshow(ch_arr, cmap='gray')
    for x in range(0, 224, 16):
        axes[2, i].axvline(x, color='red', linewidth=0.3)
        axes[2, i].axhline(x, color='red', linewidth=0.3)
    axes[2, i].set_title(f"14x14 patch grid")
    axes[2, i].axis('off')

# Also show individual channels for one cell
fig2, axes2 = plt.subplots(1, 5, figsize=(15, 3))
img = tifffile.imread(files[0]).astype(np.float32)
channels = ["647", "Brightfield", "DAPI", "488", "594"]
for c in range(5):
    axes2[c].imshow(img[:, :, c], cmap='gray')
    axes2[c].set_title(channels[c])
    axes2[c].axis('off')

plt.suptitle("Single cell: 5 channels")
plt.tight_layout()
plt.savefig("/content/cell_visualization.png", dpi=150)
plt.show()

# Print intensity statistics
print("\nIntensity stats per channel:")
for c in range(5):
    vals = img[:, :, c].flatten()
    print(f"  {channels[c]:12s}: min={vals.min():.0f} max={vals.max():.0f} mean={vals.mean():.0f} std={vals.std():.0f}")

In [ ]:
# Explore the full structure
!find /content/severin_data -type d | sort | head -40

In [ ]:
!find /content/severin_data -type f -name "*.tif" | wc -l

In [ ]:
!find /content/severin_data -type f -name "*.tiff" | wc -l


In [ ]:
!find /content/severin_data -type f -name "*.png" | wc -l


In [ ]:
!find /content/severin_data -type f | head -5


In [ ]:
!du -sh /content/severin_data/


In [ ]:
import glob

# Find an actual file
files = glob.glob("/content/severin_data/**/*.tiff", recursive=True)
print(f"Total files: {len(files)}")
print(f"First file: {files[0]}")

# Read it
import tifffile
import numpy as np

sample = tifffile.imread(files[0])
print(f"\nShape: {sample.shape}")
print(f"Dtype: {sample.dtype}")
print(f"Min: {sample.min()}, Max: {sample.max()}")

# Check splits
import os
base = "/content/severin_data/DeepPhenotype_PBMC_ImageSet_YSeverin"
print(f"\nTop-level folders: {os.listdir(base)}")

# Count per cell type
cell_counts = {}
for f in files:
    parts = f.split("/")
    cell_type = parts[-2]  # folder containing the file = cell type
    cell_counts[cell_type] = cell_counts.get(cell_type, 0) + 1

print("\nCell type counts:")
for ct, n in sorted(cell_counts.items(), key=lambda x: -x[1]):
    print(f"  {ct}: {n:,}")

In [ ]:
!cat "/content/severin_data/DeepPhenotype_PBMC_ImageSet_YSeverin/Data_Description.txt"

In [ ]:
import glob

base = "/content/severin_data/DeepPhenotype_PBMC_ImageSet_YSeverin"

train_files = glob.glob(f"{base}/Training/**/*.tiff", recursive=True)
test_files = glob.glob(f"{base}/Test/**/*.tiff", recursive=True)

print(f"Training: {len(train_files):,}")
print(f"Test: {len(test_files):,}")

# Training folder structure
import os
print(f"\nTraining subfolders: {os.listdir(os.path.join(base, 'Training'))}")

# Training cell type counts
train_counts = {}
for f in train_files:
    cell_type = f.split("/")[-2]
    train_counts[cell_type] = train_counts.get(cell_type, 0) + 1

print("\nTraining cell type counts:")
for ct, n in sorted(train_counts.items(), key=lambda x: -x[1]):
    print(f"  {ct}: {n:,}")

In [ ]:
# Check scDINO's training config for image size
!pip install -q requests
import requests

url = "https://raw.githubusercontent.com/JacobHanimann/scDINO/master/configs/scDINO_full_pipeline.yaml"
r = requests.get(url)
print(r.text)

In [1]:
# Brick 2: Data loader for Severin PBMC dataset

import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import tifffile
from PIL import Image

class SeverinDataset(Dataset):
    """Load 5-channel TIFF cell crops from the Severin PBMC dataset."""

    def __init__(self, root_dir, image_size=224):
        self.image_size = image_size
        self.files = sorted(glob.glob(os.path.join(root_dir, "**/*.tiff"), recursive=True))

        # Extract cell type labels from folder structure
        self.cell_types = sorted(list(set(
            f.split("/")[-2] for f in self.files
        )))
        self.label_map = {ct: i for i, ct in enumerate(self.cell_types)}
        self.labels = [self.label_map[f.split("/")[-2]] for f in self.files]

        print(f"Loaded {len(self.files)} images, {len(self.cell_types)} classes: {self.cell_types}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Load 5-channel TIFF (50, 50, 5) uint16
        img = tifffile.imread(self.files[idx])  # (50, 50, 5)

        # Convert to float32 and normalize to [0, 1]
        img = img.astype(np.float32)
        img = img / img.max() if img.max() > 0 else img  # per-image normalize

        # Transpose to (C, H, W) = (5, 50, 50)
        img = img.transpose(2, 0, 1)

        # Resize each channel to 224x224
        channels_resized = []
        for c in range(img.shape[0]):
            ch = Image.fromarray(img[c])
            ch = ch.resize((self.image_size, self.image_size), Image.BILINEAR)
            channels_resized.append(np.array(ch))

        img = np.stack(channels_resized, axis=0)  # (5, 224, 224)
        img = torch.from_numpy(img)

        label = self.labels[idx]
        return img, label


# Test it
base = "/content/severin_data/DeepPhenotype_PBMC_ImageSet_YSeverin"

train_dataset = SeverinDataset(os.path.join(base, "Training"), image_size=224)
test_dataset = SeverinDataset(os.path.join(base, "Test"), image_size=224)

# Check one sample
img, label = train_dataset[0]
print(f"\nSample image shape: {img.shape}")
print(f"Sample dtype: {img.dtype}")
print(f"Sample min/max: {img.min():.4f} / {img.max():.4f}")
print(f"Label: {label} ({train_dataset.cell_types[label]})")

# Test a DataLoader
loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
batch_imgs, batch_labels = next(iter(loader))
print(f"\nBatch shape: {batch_imgs.shape}")
print(f"Labels shape: {batch_labels.shape}")
print(f"Batch min/max: {batch_imgs.min():.4f} / {batch_imgs.max():.4f}")

Loaded 0 images, 0 classes: []
Loaded 0 images, 0 classes: []


IndexError: list index out of range

In [2]:
%%writefile /content/severin_dataset.py
"""Data loader for Severin PBMC 5-channel TIFF dataset."""

import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset
import tifffile
from PIL import Image


class SeverinDataset(Dataset):
    def __init__(self, root_dir, image_size=224):
        self.image_size = image_size
        self.files = sorted(glob.glob(os.path.join(root_dir, "**/*.tiff"), recursive=True))

        self.cell_types = sorted(list(set(
            f.split("/")[-2] for f in self.files
        )))
        self.label_map = {ct: i for i, ct in enumerate(self.cell_types)}
        self.labels = [self.label_map[f.split("/")[-2]] for f in self.files]

        print(f"Loaded {len(self.files)} images, {len(self.cell_types)} classes: {self.cell_types}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = tifffile.imread(self.files[idx])  # (50, 50, 5) uint16
        img = img.astype(np.float32)
        img = img / img.max() if img.max() > 0 else img
        img = img.transpose(2, 0, 1)  # (5, 50, 50)

        channels_resized = []
        for c in range(img.shape[0]):
            ch = Image.fromarray(img[c])
            ch = ch.resize((self.image_size, self.image_size), Image.BILINEAR)
            channels_resized.append(np.array(ch))

        img = np.stack(channels_resized, axis=0)  # (5, 224, 224)
        img = torch.from_numpy(img)

        return img, self.labels[idx]

Writing /content/severin_dataset.py


In [ ]:
# Brick 3a: Download DINO pretrained ViT-S/16
!wget -O /content/dino_vits16_pretrain.pth "https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth"
!ls -lh /content/dino_vits16_pretrain.pth

In [ ]:
import torch

# Load and inspect
ckpt = torch.load("/content/dino_vits16_pretrain.pth", map_location="cpu")

# What's inside?
print(f"Type: {type(ckpt)}")
if isinstance(ckpt, dict):
    print(f"Keys: {list(ckpt.keys())[:10]}")

    # Find the patch embedding (this is what we need to change from 3→5 channels)
    for key, val in ckpt.items():
        if "patch" in key or "proj" in key or "embed" in key:
            print(f"  {key}: {val.shape}")
        elif "cls" in key or "pos" in key:
            print(f"  {key}: {val.shape}")

In [ ]:
import torch
import torch.nn as nn
import timm

# Create a ViT-S/16 with timm (same architecture as DINO)
model = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)

# Load DINO pretrained weights
ckpt = torch.load("/content/dino_vits16_pretrain.pth", map_location="cpu")
msg = model.load_state_dict(ckpt, strict=False)
print(f"Loaded DINO weights. Missing: {msg.missing_keys}, Unexpected: {msg.unexpected_keys}")

# Check current patch embedding
print(f"\nOriginal patch_embed: {model.patch_embed.proj.weight.shape}")  # (384, 3, 16, 16)

# === Adapt from 3 to 5 channels ===
old_proj = model.patch_embed.proj  # Conv2d(3, 384, kernel_size=16, stride=16)
old_weight = old_proj.weight.data  # (384, 3, 16, 16)

# Create new Conv2d with 5 input channels
new_proj = nn.Conv2d(5, 384, kernel_size=16, stride=16)

# Initialize: copy the 3 pretrained channels, average them for the 2 new channels
new_weight = torch.zeros(384, 5, 16, 16)
new_weight[:, :3, :, :] = old_weight  # copy existing 3 channels
new_weight[:, 3, :, :] = old_weight.mean(dim=1)  # channel 4 = average of RGB
new_weight[:, 4, :, :] = old_weight.mean(dim=1)  # channel 5 = average of RGB

new_proj.weight = nn.Parameter(new_weight)
new_proj.bias = old_proj.bias  # keep bias

model.patch_embed.proj = new_proj
print(f"New patch_embed: {model.patch_embed.proj.weight.shape}")  # (384, 5, 16, 16)

# Test forward pass with 5-channel input
dummy = torch.randn(2, 5, 224, 224)
with torch.no_grad():
    features = model.forward_features(dummy)
    print(f"\nOutput shape: {features.shape}")
    cls_token = features[:, 0]
    print(f"CLS token shape: {cls_token.shape}")
    patch_tokens = features[:, 1:]
    print(f"Patch tokens shape: {patch_tokens.shape}")

In [3]:
%%writefile /content/cell_jepa.py
"""Cell-JEPA: Spatial masked prediction + SIGReg for cell microscopy."""

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import math


class SIGReg(nn.Module):
    """From LeWM — forces embeddings toward Gaussian distribution."""
    def __init__(self, knots=17, num_proj=1024):
        super().__init__()
        self.num_proj = num_proj
        t = torch.linspace(0, 3, knots, dtype=torch.float32)
        dt = 3 / (knots - 1)
        weights = torch.full((knots,), 2 * dt, dtype=torch.float32)
        weights[[0, -1]] = dt
        window = torch.exp(-t.square() / 2.0)
        self.register_buffer("t", t)
        self.register_buffer("phi", window)
        self.register_buffer("weights", weights * window)

    def forward(self, proj):
        device = proj.device
        A = torch.randn(proj.size(-1), self.num_proj, device=device)
        A = A.div_(A.norm(p=2, dim=0))
        x_t = (proj @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        statistic = (err @ self.weights) * proj.size(-2)
        return statistic.mean()


class SpatialPredictor(nn.Module):
    """Predicts masked patch embeddings from visible ones."""
    def __init__(self, embed_dim=384, depth=4, num_heads=6, mlp_ratio=4.0, num_patches=196):
        super().__init__()
        self.mask_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, embed_dim) * 0.02)

        # Simple transformer blocks (no action conditioning, non-causal)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=num_heads,
                dim_feedforward=int(embed_dim * mlp_ratio),
                dropout=0.0,
                activation='gelu',
                batch_first=True,
                norm_first=True,
            )
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, visible_emb, mask_indices, visible_indices, total_patches):
        B, _, D = visible_emb.shape

        # Build full sequence: mask tokens everywhere, then insert visible tokens
        full_seq = self.mask_token.expand(B, total_patches, -1).clone()
        full_seq[:, visible_indices] = visible_emb
        full_seq = full_seq + self.pos_embed[:, :total_patches]

        # Run through transformer (non-causal, all attend to all)
        for block in self.blocks:
            full_seq = block(full_seq)
        full_seq = self.norm(full_seq)

        # Return only predicted masked positions
        return full_seq[:, mask_indices]


class CellJEPA(nn.Module):
    """Full Cell-JEPA: encoder + spatial predictor."""
    def __init__(self, encoder, predictor, mask_ratio=0.6):
        super().__init__()
        self.encoder = encoder
        self.predictor = predictor
        self.mask_ratio = mask_ratio

    def random_mask(self, n_patches, device):
        n_mask = int(n_patches * self.mask_ratio)
        perm = torch.randperm(n_patches, device=device)
        return perm[:n_mask], perm[n_mask:]  # mask_indices, visible_indices

    def forward(self, images):
        B = images.size(0)

        # Encode full image (all patches)
        features = self.encoder.forward_features(images)  # (B, 1+N, 384)
        cls_token = features[:, 0]        # (B, 384)
        patch_tokens = features[:, 1:]    # (B, 196, 384)
        N = patch_tokens.size(1)

        # Generate mask
        mask_idx, visible_idx = self.random_mask(N, images.device)

        # Targets: masked patch embeddings (stop gradient!)
        target_emb = patch_tokens[:, mask_idx].detach()  # (B, n_mask, 384)

        # Input to predictor: visible patch embeddings
        visible_emb = patch_tokens[:, visible_idx]  # (B, n_visible, 384)

        # Predict masked patches
        pred_emb = self.predictor(visible_emb, mask_idx, visible_idx, N)

        return {
            "pred_emb": pred_emb,       # (B, n_mask, 384)
            "target_emb": target_emb,   # (B, n_mask, 384)
            "cls_token": cls_token,      # (B, 384) for downstream eval
            "all_patches": patch_tokens, # (B, 196, 384) for SIGReg
        }


def build_cell_jepa(checkpoint_path, in_channels=5, mask_ratio=0.6, predictor_depth=4):
    """Build Cell-JEPA from a DINO pretrained ViT-S/16."""

    # Load ViT-S/16
    encoder = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    encoder.load_state_dict(ckpt, strict=False)

    # Adapt to N channels
    if in_channels != 3:
        old_proj = encoder.patch_embed.proj
        old_weight = old_proj.weight.data
        new_proj = nn.Conv2d(in_channels, 384, kernel_size=16, stride=16)
        new_weight = torch.zeros(384, in_channels, 16, 16)
        new_weight[:, :3, :, :] = old_weight
        for c in range(3, in_channels):
            new_weight[:, c, :, :] = old_weight.mean(dim=1)
        new_proj.weight = nn.Parameter(new_weight)
        new_proj.bias = old_proj.bias
        encoder.patch_embed.proj = new_proj

    # Build predictor
    predictor = SpatialPredictor(
        embed_dim=384,
        depth=predictor_depth,
        num_heads=6,
        num_patches=196,
    )

    model = CellJEPA(encoder, predictor, mask_ratio=mask_ratio)
    return model

Writing /content/cell_jepa.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/cell_jepa_checkpoints", exist_ok=True)

In [4]:
%%writefile /content/cell_jepa_train.py
"""Train Cell-JEPA on Severin PBMC dataset."""

import os
import sys
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

sys.path.insert(0, '/content')
from severin_dataset import SeverinDataset
from cell_jepa import build_cell_jepa, SIGReg
import shutil

def train():
    device = torch.device("cuda")

    # === Config ===
    batch_size = 24
    epochs = 50
    lr = 1e-4
    weight_decay = 0.05
    sigreg_weight = 10.0
    mask_ratio = 0.6
    seed = 42
    n_images = 5000
    # === Data ===
    base = "/content/severin_data/DeepPhenotype_PBMC_ImageSet_YSeverin"
    train_dataset = SeverinDataset(os.path.join(base, "Training"), image_size=224)
    test_dataset = SeverinDataset(os.path.join(base, "Test"), image_size=224)

    # trying only a subset of the dataset for computation limitation #
    subset_idx = torch.randperm(len(train_dataset))[:n_images]
    train_subset = torch.utils.data.Subset(train_dataset, subset_idx)

    #train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
    #                          drop_last=True, num_workers=2, pin_memory=True)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,
                              drop_last=True, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=0, pin_memory=True)

    # === Model ===
    model = build_cell_jepa(
        checkpoint_path="/content/dino_vits16_pretrain.pth",
        in_channels=5,
        mask_ratio=mask_ratio,
        predictor_depth=4,
    ).to(device)

    sigreg = SIGReg(knots=17, num_proj=1024).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    encoder_params = sum(p.numel() for p in model.encoder.parameters())
    predictor_params = sum(p.numel() for p in model.predictor.parameters())
    print(f"Total params: {total_params:,}")
    print(f"  Encoder: {encoder_params:,}")
    print(f"  Predictor: {predictor_params:,}")

    # === Optimizer ===
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # === Training ===
    output_dir = "/content/cell_jepa_output"
    os.makedirs(output_dir, exist_ok=True)
    best_val_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        train_pred_loss = 0.0
        train_sigreg_loss = 0.0
        n_batches = 0

        for images, labels in train_loader:
            images = images.to(device)

            output = model(images)

            # Prediction loss: MSE between predicted and target patch embeddings
            pred_loss = F.mse_loss(output["pred_emb"], output["target_emb"])

            # SIGReg on CLS tokens
            cls_for_sigreg = output["cls_token"].unsqueeze(0)  # (1, B, 384)
            sig_loss = sigreg(cls_for_sigreg)

            loss = pred_loss + sigreg_weight * sig_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_pred_loss += pred_loss.item()
            train_sigreg_loss += sig_loss.item()
            n_batches += 1

        scheduler.step()

        # === Validation ===
        model.eval()
        val_pred_loss = 0.0
        val_sigreg_loss = 0.0
        val_batches = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                output = model(images)
                pred_loss = F.mse_loss(output["pred_emb"], output["target_emb"])
                sig_loss = sigreg(output["cls_token"].unsqueeze(0))
                val_pred_loss += pred_loss.item()
                val_sigreg_loss += sig_loss.item()
                val_batches += 1

        avg_train_pred = train_pred_loss / n_batches
        avg_train_sig = train_sigreg_loss / n_batches
        avg_val_pred = val_pred_loss / val_batches
        avg_val_sig = val_sigreg_loss / val_batches

        print(f"Epoch {epoch+1:3d}/{epochs} | "
              f"Train pred:{avg_train_pred:.4f} sig:{avg_train_sig:.4f} | "
              f"Val pred:{avg_val_pred:.4f} sig:{avg_val_sig:.4f} | "
              f"LR:{scheduler.get_last_lr()[0]:.6f}")

        # Save best
        val_total = avg_val_pred + sigreg_weight * avg_val_sig
        if val_total < best_val_loss:
            best_val_loss = val_total
            torch.save(model.state_dict(), os.path.join(output_dir, "best_model.pt"))
            shutil.copy(os.path.join(output_dir, "best_model.pt"), f"/content/drive/MyDrive/cell_jepa_checkpoints/best_{n_images}_{epoch+1}ep.pt")
            print(f"  -> Saved best model ")

        # Checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), os.path.join(output_dir, f"epoch_{epoch+1}.pt"))

    torch.save(model.state_dict(), os.path.join(output_dir, "final_model.pt"))
    print(f"Done. Models saved to {output_dir}")


if __name__ == "__main__":
    train()

Writing /content/cell_jepa_train.py


In [ ]:
!python /content/cell_jepa_train.py

**K-NN evaluation**

**Plot K-NN and t-SNE from training**

In [ ]:
!wget -q -O /content/dino_vits16.pth "https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth"

In [ ]:
import os, glob, torch, timm, numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tifffile
from PIL import Image
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# --- Dataset ---
class SeverinDataset(Dataset):
    def __init__(self, root_dir, image_size=224):
        self.image_size = image_size
        self.files = sorted(glob.glob(os.path.join(root_dir, "**/*.tiff"), recursive=True))
        self.cell_types = sorted(list(set(f.split("/")[-2] for f in self.files)))
        self.label_map = {ct: i for i, ct in enumerate(self.cell_types)}
        self.labels = [self.label_map[f.split("/")[-2]] for f in self.files]
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        img = tifffile.imread(self.files[idx]).astype(np.float32)
        img = img / img.max() if img.max() > 0 else img
        img = img.transpose(2, 0, 1)
        channels = []
        for c in range(img.shape[0]):
            ch = Image.fromarray(img[c])
            ch = ch.resize((self.image_size, self.image_size), Image.BILINEAR)
            channels.append(np.array(ch))
        return torch.from_numpy(np.stack(channels)), self.labels[idx]

# --- Build encoder ---
def make_5ch_encoder():
    encoder = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
    ckpt = torch.load("/content/dino_vits16.pth", map_location="cpu")
    encoder.load_state_dict(ckpt, strict=False)
    old_w = encoder.patch_embed.proj.weight.data
    new_proj = nn.Conv2d(5, 384, kernel_size=16, stride=16)
    new_w = torch.zeros(384, 5, 16, 16)
    new_w[:, :3] = old_w
    new_w[:, 3] = old_w.mean(dim=1)
    new_w[:, 4] = old_w.mean(dim=1)
    new_proj.weight = nn.Parameter(new_w)
    new_proj.bias = encoder.patch_embed.proj.bias
    encoder.patch_embed.proj = new_proj
    return encoder

# --- Feature extraction ---
def extract_features(encoder, loader, device):
    encoder.eval()
    all_feats, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            features = encoder.forward_features(images.to(device))
            all_feats.append(features[:, 0].cpu())
            all_labels.extend(labels)
    return torch.cat(all_feats).numpy(), np.array(all_labels)

device = torch.device("cuda")
base = "/content/severin_data/DeepPhenotype_PBMC_ImageSet_YSeverin"

train_dataset = SeverinDataset(os.path.join(base, "Training"))
test_dataset = SeverinDataset(os.path.join(base, "Test"))
train_loader = DataLoader(train_dataset, batch_size=24, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=24, shuffle=False, num_workers=0)

# === BASELINE: DINO pretrained, zero cell training ===
print("Loading baseline encoder...")
baseline_encoder = make_5ch_encoder().to(device)

# === CELL-JEPA: Load trained checkpoint ===
print("Loading Cell-JEPA encoder...")
jepa_encoder = make_5ch_encoder()
# Load full model state dict, extract just encoder keys
full_state = torch.load("/content/cell_jepa_output/best_model.pt", map_location="cpu")
encoder_state = {k.replace("encoder.", ""): v for k, v in full_state.items() if k.startswith("encoder.")}
if len(encoder_state) == 0:
    # Maybe the state dict IS just the encoder
    encoder_state = full_state
jepa_encoder.load_state_dict(encoder_state, strict=False)
jepa_encoder = jepa_encoder.to(device)

# === Extract features ===
print("Extracting test features (baseline)...")
baseline_test_feats, test_labels = extract_features(baseline_encoder, test_loader, device)
print("Extracting test features (Cell-JEPA)...")
jepa_test_feats, _ = extract_features(jepa_encoder, test_loader, device)
print("Extracting train features (baseline)...")
baseline_train_feats, train_labels = extract_features(baseline_encoder, train_loader, device)
print("Extracting train features (Cell-JEPA)...")
jepa_train_feats, _ = extract_features(jepa_encoder, train_loader, device)

# === k-NN ===
print("\n=== k-NN Classification Results ===")
for k in [5, 10, 20, 50]:
    acc_base = accuracy_score(test_labels, KNeighborsClassifier(n_neighbors=k).fit(baseline_train_feats, train_labels).predict(baseline_test_feats))
    acc_jepa = accuracy_score(test_labels, KNeighborsClassifier(n_neighbors=k).fit(jepa_train_feats, train_labels).predict(jepa_test_feats))
    print(f"k={k:3d} | Baseline: {acc_base:.4f} | Cell-JEPA: {acc_jepa:.4f} | Delta: {acc_jepa-acc_base:+.4f}")

# === t-SNE ===
print("\nComputing t-SNE...")
n = 5000
pca_base = PCA(n_components=50).fit_transform(baseline_test_feats[:n])
pca_jepa = PCA(n_components=50).fit_transform(jepa_test_feats[:n])
tsne_base = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(pca_base)
tsne_jepa = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(pca_jepa)

labels_n = test_labels[:n]
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for i, ct in enumerate(test_dataset.cell_types):
    mask = labels_n == i
    axes[0].scatter(tsne_base[mask, 0], tsne_base[mask, 1], s=3, alpha=0.5, label=ct)
    axes[1].scatter(tsne_jepa[mask, 0], tsne_jepa[mask, 1], s=3, alpha=0.5, label=ct)
axes[0].set_title('Baseline: DINO ViT-S/16 (no cell training)')
axes[1].set_title('Cell-JEPA: JEPA + SIGReg')
axes[0].legend(markerscale=5)
axes[1].legend(markerscale=5)
plt.tight_layout()
plt.savefig('/content/cell_jepa_vs_baseline.png', dpi=150)
plt.show()
print("DONE!")

**Plot K-NN and t-SNE loading from mounted gdrive**

In [ ]:
import os, glob, torch, timm, numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tifffile
from PIL import Image
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# --- Dataset ---
class SeverinDataset(Dataset):
    def __init__(self, root_dir, image_size=224):
        self.image_size = image_size
        self.files = sorted(glob.glob(os.path.join(root_dir, "**/*.tiff"), recursive=True))
        self.cell_types = sorted(list(set(f.split("/")[-2] for f in self.files)))
        self.label_map = {ct: i for i, ct in enumerate(self.cell_types)}
        self.labels = [self.label_map[f.split("/")[-2]] for f in self.files]
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        img = tifffile.imread(self.files[idx]).astype(np.float32)
        img = img / img.max() if img.max() > 0 else img
        img = img.transpose(2, 0, 1)
        channels = []
        for c in range(img.shape[0]):
            ch = Image.fromarray(img[c])
            ch = ch.resize((self.image_size, self.image_size), Image.BILINEAR)
            channels.append(np.array(ch))
        return torch.from_numpy(np.stack(channels)), self.labels[idx]

# --- Build encoder ---
def make_5ch_encoder():
    encoder = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
    ckpt = torch.load("/content/dino_vits16.pth", map_location="cpu")
    encoder.load_state_dict(ckpt, strict=False)
    old_w = encoder.patch_embed.proj.weight.data
    new_proj = nn.Conv2d(5, 384, kernel_size=16, stride=16)
    new_w = torch.zeros(384, 5, 16, 16)
    new_w[:, :3] = old_w
    new_w[:, 3] = old_w.mean(dim=1)
    new_w[:, 4] = old_w.mean(dim=1)
    new_proj.weight = nn.Parameter(new_w)
    new_proj.bias = encoder.patch_embed.proj.bias
    encoder.patch_embed.proj = new_proj
    return encoder

# --- Feature extraction ---
def extract_features(encoder, loader, device):
    encoder.eval()
    all_feats, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            features = encoder.forward_features(images.to(device))
            all_feats.append(features[:, 0].cpu())
            all_labels.extend(labels)
    return torch.cat(all_feats).numpy(), np.array(all_labels)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base = "/content/severin_data/DeepPhenotype_PBMC_ImageSet_YSeverin"

train_dataset = SeverinDataset(os.path.join(base, "Training"))
test_dataset = SeverinDataset(os.path.join(base, "Test"))
train_loader = DataLoader(train_dataset, batch_size=24, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=24, shuffle=False, num_workers=0)

# === BASELINE: DINO pretrained, zero cell training ===
print("Loading baseline encoder...")
baseline_encoder = make_5ch_encoder().to(device)

# === CELL-JEPA: Load trained checkpoint ===

# Replace this section:
print("Loading Cell-JEPA encoder...")
jepa_encoder = make_5ch_encoder()
full_state = torch.load("/content/drive/MyDrive/cell_jepa_checkpoints/best_5000k_24ep.pt", map_location="cpu")

# Debug: see what keys look like
print(f"Total keys: {len(full_state)}")
print(f"First 5 keys: {list(full_state.keys())[:5]}")

encoder_state = {k.replace("encoder.", ""): v for k, v in full_state.items() if k.startswith("encoder.")}
print(f"Encoder keys found: {len(encoder_state)}")

if len(encoder_state) == 0:
    print("No 'encoder.' prefix found — loading state dict directly")
    encoder_state = full_state

msg = jepa_encoder.load_state_dict(encoder_state, strict=False)
print(f"Missing keys: {len(msg.missing_keys)}")
print(f"Unexpected keys: {len(msg.unexpected_keys)}")
if msg.missing_keys:
    print(f"First missing: {msg.missing_keys[:3]}")
if msg.unexpected_keys:
    print(f"First unexpected: {msg.unexpected_keys[:3]}")

jepa_encoder = jepa_encoder.to(device)

# === Extract features ===
print("Extracting test features (baseline)...")
baseline_test_feats, test_labels = extract_features(baseline_encoder, test_loader, device)
print("Extracting test features (Cell-JEPA)...")
jepa_test_feats, _ = extract_features(jepa_encoder, test_loader, device)
print("Extracting train features (baseline)...")
baseline_train_feats, train_labels = extract_features(baseline_encoder, train_loader, device)
print("Extracting train features (Cell-JEPA)...")
jepa_train_feats, _ = extract_features(jepa_encoder, train_loader, device)

# === k-NN ===
print("\n=== k-NN Classification Results ===")
for k in [5, 10, 20, 50]:
    acc_base = accuracy_score(test_labels, KNeighborsClassifier(n_neighbors=k).fit(baseline_train_feats, train_labels).predict(baseline_test_feats))
    acc_jepa = accuracy_score(test_labels, KNeighborsClassifier(n_neighbors=k).fit(jepa_train_feats, train_labels).predict(jepa_test_feats))
    print(f"k={k:3d} | Baseline: {acc_base:.4f} | Cell-JEPA: {acc_jepa:.4f} | Delta: {acc_jepa-acc_base:+.4f}")

# === t-SNE ===
print("\nComputing t-SNE...")
n = 5000
pca_base = PCA(n_components=50).fit_transform(baseline_test_feats[:n])
pca_jepa = PCA(n_components=50).fit_transform(jepa_test_feats[:n])
tsne_base = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(pca_base)
tsne_jepa = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(pca_jepa)

labels_n = test_labels[:n]
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for i, ct in enumerate(test_dataset.cell_types):
    mask = labels_n == i
    axes[0].scatter(tsne_base[mask, 0], tsne_base[mask, 1], s=3, alpha=0.5, label=ct)
    axes[1].scatter(tsne_jepa[mask, 0], tsne_jepa[mask, 1], s=3, alpha=0.5, label=ct)
axes[0].set_title('Baseline: DINO ViT-S/16 (no cell training)')
axes[1].set_title('Cell-JEPA: JEPA + SIGReg (24 epochs)')
axes[0].legend(markerscale=5)
axes[1].legend(markerscale=5)
plt.tight_layout()
plt.savefig('/content/cell_jepa_vs_baseline.png', dpi=150)
plt.show()
print("DONE!")

**Fig. Plot**

In [ ]:
# This needs NO GPU — just matplotlib
# Run in Colab with CPU runtime or locally

import matplotlib.pyplot as plt
import numpy as np

# === DATA FROM YOUR TRAINING LOGS ===

# 1K images, 30 epochs (complete)
ep_1k = list(range(1, 31))
pred_train_1k = [12.36, 6.28, 4.46, 3.70, 3.33, 2.91, 2.67, 2.62, 2.39, 2.44,
                 2.09, 2.23, 1.79, 2.09, 1.79, 1.95, 1.92, 1.98, 1.70, 1.90,
                 1.68, 1.74, 1.68, 1.73, 1.71, 1.69, 1.74, 1.71, 1.70, 1.71]
pred_val_1k = [7.65, 4.45, 3.95, 3.49, 3.49, 2.89, 2.91, 2.81, 2.39, 2.69,
               2.71, 2.05, 2.04, 2.14, 1.99, 2.02, 2.58, 1.66, 1.81, 1.71,
               1.83, 1.75, 1.80, 1.84, 1.73, 1.79, 1.82, 1.75, 1.78, 1.78]
sig_train_1k = [23.42, 16.65, 14.22, 11.79, 10.94, 10.42, 10.19, 9.75, 9.35, 9.33,
                9.65, 9.00, 8.97, 9.00, 8.65, 8.54, 8.45, 8.33, 8.21, 8.10,
                8.06, 7.95, 8.02, 7.84, 7.91, 7.71, 7.80, 7.62, 7.60, 7.68]

# 5K images, 25 epochs (second attempt, model saved)
ep_5k = list(range(1, 25))
pred_train_5k = [5.70, 2.40, 1.96, 1.83, 1.67, 1.48, 1.43, 1.52, 1.35, 1.38,
                 1.30, 1.27, 1.17, 1.14, 1.08, 1.02, 1.03, 0.99, 0.97, 0.81,
                 0.89, 0.66, 0.67, 0.71]
pred_val_5k = [3.19, 2.47, 2.16, 2.15, 1.75, 1.71, 1.68, 1.74, 1.73, 1.36,
               1.53, 1.34, 1.54, 1.55, 1.26, 1.43, 0.99, 1.25, 1.38, 0.79,
               0.79, 0.71, 0.91, 0.67]
sig_train_5k = [15.78, 10.09, 8.95, 8.58, 8.16, 7.66, 7.39, 7.40, 7.10, 6.92,
                6.66, 6.50, 6.40, 6.21, 6.01, 5.90, 5.81, 5.54, 5.45, 5.33,
                5.26, 5.12, 5.05, 4.85]

# 40K images, 8 epochs
ep_40k = list(range(1, 9))
pred_train_40k = [1.98, 0.79, 0.49, 0.39, 0.36, 0.41, 0.42, 0.52]
pred_val_40k = [0.98, 0.74, 0.39, 0.32, 0.39, 0.47, 0.38, 0.32]
sig_train_40k = [9.23, 6.55, 5.07, 3.89, 3.02, 2.42, 2.02, 2.06]

# === FIGURE 1: Training curves comparison ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Prediction loss
axes[0].plot(ep_1k, pred_val_1k, 'o-', markersize=2, label='1K images', color='#e74c3c')
axes[0].plot(ep_5k, pred_val_5k, 's-', markersize=2, label='5K images', color='#3498db')
axes[0].plot(ep_40k, pred_val_40k, '^-', markersize=3, label='40K images', color='#2ecc71')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Prediction Loss (MSE)')
axes[0].set_title('Prediction Loss')
axes[0].legend()
axes[0].set_ylim(0, 8)
axes[0].grid(True, alpha=0.3)

# SIGReg loss
axes[1].plot(ep_1k, sig_train_1k, 'o-', markersize=2, label='1K images', color='#e74c3c')
axes[1].plot(ep_5k, sig_train_5k, 's-', markersize=2, label='5K images', color='#3498db')
axes[1].plot(ep_40k, sig_train_40k, '^-', markersize=3, label='40K images', color='#2ecc71')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('SIGReg Loss (train)')
axes[1].set_title('SIGReg Regularization Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Cell-JEPA Training Dynamics Across Dataset Sizes', fontsize=14)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=200, bbox_inches='tight')
plt.show()

# === FIGURE 2: k-NN results bar chart ===
fig, ax = plt.subplots(figsize=(12, 6))

k_values = [5, 10, 20, 50]
baseline = [0.5316, 0.5495, 0.5614, 0.5608]
jepa_1k = [0.6761, 0.6726, 0.6692, 0.6510]
jepa_5k = [0.6397, 0.6389, 0.6335, 0.6174]
jepa_40k = [0.6379, 0.6361, 0.6293, 0.6069]

x = np.arange(len(k_values))
width = 0.2

bars1 = ax.bar(x - 1.5*width, baseline, width, label='Baseline (DINO, no cell training)', color='#95a5a6')
bars2 = ax.bar(x - 0.5*width, jepa_1k, width, label='Cell-JEPA (1K images, 30 epochs)', color='#e74c3c')
bars3 = ax.bar(x + 0.5*width, jepa_5k, width, label='Cell-JEPA (5K images, 24 epochs)', color='#3498db')
bars4 = ax.bar(x + 1.5*width, jepa_40k, width, label='Cell-JEPA (40K images, 8 epochs)', color='#2ecc71')

ax.set_xlabel('k (number of neighbors)', fontsize=12)
ax.set_ylabel('Classification Accuracy', fontsize=12)
ax.set_title('k-NN Classification on Severin PBMC Test Set (24,000 cells, 8 classes)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels([f'k={k}' for k in k_values])
ax.legend(loc='upper right')
ax.set_ylim(0.4, 0.75)
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1%}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('knn_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

# === FIGURE 3: Data efficiency ===
fig, ax = plt.subplots(figsize=(8, 5))

sizes = [0, 1000, 5000, 40000]
acc_k5 = [0.5316, 0.6761, 0.6397, 0.6379]
labels_pts = [
    'Baseline\n(0 images)',
    '1K images\n(30 ep, converged)',
    '5K images\n(24 ep, incomplete)',
    '40K images\n(8 ep, incomplete)'
]

ax.plot(sizes, acc_k5, 'o-', markersize=10, color='#2c3e50', linewidth=2)

offsets = [(10, 20), (10, 20), (10, -30), (10, -30)]
for i, (s, a, l, off) in enumerate(zip(sizes, acc_k5, labels_pts, offsets)):
    ax.annotate(f'{a:.1%}\n{l}',
                xy=(s, a),
                xytext=off,
                textcoords="offset points",
                fontsize=9,
                arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_xlabel('Number of Training Images', fontsize=12)
ax.set_ylabel('k-NN Accuracy (k=5)', fontsize=12)
ax.set_title('Cell-JEPA: Data Efficiency on Severin PBMC', fontsize=13)
ax.set_ylim(0.45, 0.75)
ax.grid(True, alpha=0.3)
ax.axhline(y=0.5316, color='gray', linestyle='--', alpha=0.5, label='Zero-shot baseline')
ax.legend()

plt.tight_layout()
plt.savefig('data_efficiency.png', dpi=200, bbox_inches='tight')
plt.show()

print("All figures saved!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Upload your 3 t-SNE images to Colab first (drag into file browser)
# Or if they're already saved:

fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# Top left: Baseline (same in all comparisons, use any one)
# We'll create a text-only panel instead
axes[0, 0].text(0.5, 0.5,
    'Cell-JEPA\n\n'
    'Spatial masked prediction\n+ SIGReg regularization\n\n'
    'No EMA\nNo augmentations\nNo multi-term loss\n\n'
    'Pretrained: DINO ViT-S/16\nChannels: 5 (fluorescence)\nDataset: Severin PBMC\n'
    'Classes: 8 immune cell types',
    transform=axes[0, 0].transAxes,
    fontsize=14, verticalalignment='center', horizontalalignment='center',
    bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8))
axes[0, 0].axis('off')
axes[0, 0].set_title('Method Summary', fontsize=14, fontweight='bold')

# Top right: k-NN results as table
cell_text = [
    ['Baseline (no training)', '53.2%', '55.0%', '56.1%'],
    ['Cell-JEPA (1K, 30 ep)', '67.6%', '67.3%', '66.9%'],
    ['Cell-JEPA (5K, 24 ep)', '64.0%', '63.9%', '63.4%'],
    ['Cell-JEPA (40K, 8 ep)', '63.8%', '63.6%', '62.9%'],
]
col_labels = ['Method', 'k=5', 'k=10', 'k=20']

axes[0, 1].axis('off')
table = axes[0, 1].table(
    cellText=cell_text,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colColours=['#d5e8f0']*4
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)
# Bold the best results
for i in range(1, 5):
    for j in range(4):
        cell = table[i, j]
        if i == 1:  # header row offset
            cell.set_facecolor('#f8f9fa')
        if i == 2:  # 1K row - best results
            cell.set_facecolor('#d4efdf')
axes[0, 1].set_title('k-NN Classification Accuracy (24,000 test cells)', fontsize=14, fontweight='bold')

# Bottom left: Data efficiency plot
sizes = [0, 1000, 5000, 40000]
acc_k5 = [0.5316, 0.6761, 0.6397, 0.6379]

axes[1, 0].plot(sizes, acc_k5, 'o-', markersize=12, color='#2c3e50', linewidth=2.5)
axes[1, 0].axhline(y=0.5316, color='gray', linestyle='--', alpha=0.5)

annotations = [
    (0, 0.5316, 'Baseline', (15, -20)),
    (1000, 0.6761, '1K (30ep)\nconverged', (15, 10)),
    (5000, 0.6397, '5K (24ep)\nincomplete', (15, -25)),
    (40000, 0.6379, '40K (8ep)\nincomplete', (-80, 15)),
]
for x, y, label, offset in annotations:
    axes[1, 0].annotate(f'{y:.1%}\n{label}', xy=(x, y), xytext=offset,
                         textcoords="offset points", fontsize=9,
                         arrowprops=dict(arrowstyle='->', color='gray'))

axes[1, 0].set_xlabel('Training Images', fontsize=12)
axes[1, 0].set_ylabel('k-NN Accuracy (k=5)', fontsize=12)
axes[1, 0].set_title('Data Efficiency', fontsize=14, fontweight='bold')
axes[1, 0].set_ylim(0.45, 0.75)
axes[1, 0].grid(True, alpha=0.3)

# Bottom right: Bar chart
import numpy as np
k_values = [5, 10, 20, 50]
baseline = [0.5316, 0.5495, 0.5614, 0.5608]
jepa_1k = [0.6761, 0.6726, 0.6692, 0.6510]
jepa_5k = [0.6397, 0.6389, 0.6335, 0.6174]
jepa_40k = [0.6379, 0.6361, 0.6293, 0.6069]

x = np.arange(len(k_values))
w = 0.2
axes[1, 1].bar(x - 1.5*w, baseline, w, label='Baseline', color='#95a5a6')
axes[1, 1].bar(x - 0.5*w, jepa_1k, w, label='1K/30ep', color='#e74c3c')
axes[1, 1].bar(x + 0.5*w, jepa_5k, w, label='5K/24ep', color='#3498db')
axes[1, 1].bar(x + 1.5*w, jepa_40k, w, label='40K/8ep', color='#2ecc71')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels([f'k={k}' for k in k_values])
axes[1, 1].set_ylabel('Accuracy', fontsize=12)
axes[1, 1].set_title('k-NN by Neighborhood Size', fontsize=14, fontweight='bold')
axes[1, 1].legend(fontsize=9)
axes[1, 1].set_ylim(0.4, 0.75)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Cell-JEPA: Preliminary Results on Severin PBMC Dataset', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('cell_jepa_summary.png', dpi=200, bbox_inches='tight')
plt.show()
print("Summary figure saved!")